# **07 - Join shows + artists + venues into one table**
***

**What:** Joins the three tables into one row per show, keeping only the columns we plan to evaluate as model features. This is the **first draft, before we drop any features**.

**Why:** The model needs everything about a show in one place: when it is, how many tickets are listed, how popular the artist is, and what kind of venue and county it is in. Each table only has part of that.

**What it does:**
1. Loads `shows.csv`, `artists.csv` and `venues.csv`.
2. Keeps only the chosen columns from each table (listed in the setup cell).
3. Joins `artists` onto `shows` on `artist_key` and `venues` onto `shows` on `venue_key` (many shows to one artist or venue, so the row count must not change).
4. Renames the two columns that clash (`country` exists in both artists and venues).
5. Checks the join (row count, missing values per column) and saves `data/joined_table.csv`.

**Where it fits:** it runs after `03_build_shows.ipynb`, `04_build_artists_table.ipynb` and `05_build_venues.ipynb` (to-do step 7). `06_cleaning_procedure.ipynb` holds the cleaning decisions (bad artist matches, non-US shows, extreme prices); they are **not applied here yet**, so this table is the uncleaned draft.

---

**Input:** `data/shows.csv`, `data/artists.csv`, `data/venues.csv`
**Output:** `data/joined_table.csv`, one row per show

## 1. Setup

In [1]:
import pandas as pd

SHOWS_PATH = "data/shows.csv"
ARTISTS_PATH = "data/artists.csv"
VENUES_PATH = "data/venues.csv"
OUT_PATH = "data/joined_table.csv"

KEEP_KEYS = False   # True = also keep artist_key and venue_key (handy for grouped cross-validation)

# Columns to keep from each table (first draft, before we drop any features)
SHOW_COLS = [
    "name",                                                              # identifier
    "weekday", "is_weekend", "month", "show_hour", "days_to_show",       # date and time
    "listings", "max_listings", "n_priced_days",                         # supply
    "price_median", "log_price_median",                                  # target (pick one later)
]
ARTIST_COLS = [
    "listeners", "log_listeners", "playcount", "plays_per_listener",     # popularity (Last.fm)
    "top_tag", "tags",                                                   # genre
    "on_tour", "n_shows", "n_venues",                                    # touring
    "artist_type", "country", "year_formed", "years_active",             # background (MusicBrainz)
    "lastfm_status", "name_similarity", "low_confidence_match",          # match quality
]
VENUE_COLS = [
    "city", "state", "country", "is_us", "lat", "lon", "timezone",       # location
    "venue_tier", "median_max_listings",                                 # size
    "sg_upcoming_events",                                                # SeatGeek
    "median_income",                                                     # county demographics
]

## 2. Load the three tables

In [2]:
shows = pd.read_csv(SHOWS_PATH, low_memory=False)
artists = pd.read_csv(ARTISTS_PATH, low_memory=False)
venues = pd.read_csv(VENUES_PATH, low_memory=False)
print(f"shows {len(shows):,} | artists {len(artists):,} | venues {len(venues):,}")

# one row per key is what makes the joins safe
assert artists["artist_key"].is_unique, "artists.csv has repeated artist_key"
assert venues["venue_key"].is_unique, "venues.csv has repeated venue_key"

shows 29,613 | artists 5,139 | venues 2,722


## 3. Pick the columns
`country` is in both `artists` (where the artist is from) and `venues` (where the show is), so they are renamed `artist_country` and `venue_country` to keep them apart. The artists' `n_shows` and `n_venues` count how many shows and venues that artist has in our data.

In [3]:
s = shows[["artist_key", "venue_key"] + SHOW_COLS]
a = artists[["artist_key"] + ARTIST_COLS].rename(columns={"country": "artist_country"})
v = venues[["venue_key"] + VENUE_COLS].rename(columns={"country": "venue_country"})
print(f"kept {len(SHOW_COLS)} show columns, {len(ARTIST_COLS)} artist columns, {len(VENUE_COLS)} venue columns")

kept 11 show columns, 16 artist columns, 11 venue columns


## 4. Join

In [4]:
joined = (s.merge(a, on="artist_key", how="left", validate="many_to_one")
           .merge(v, on="venue_key", how="left", validate="many_to_one"))

assert len(joined) == len(shows), "join changed the number of rows"
print(f"{len(joined):,} rows x {joined.shape[1]} columns (one row per show)")
print(f"shows with an artist row: {joined['listeners'].notna().mean():.0%} have Last.fm popularity")
print(f"shows with a venue row:   {joined['venue_tier'].notna().mean():.0%} have a venue tier")

29,613 rows x 40 columns (one row per show)
shows with an artist row: 94% have Last.fm popularity
shows with a venue row:   100% have a venue tier


## 5. Check missing values and save

In [5]:
if not KEEP_KEYS:
    joined = joined.drop(columns=["artist_key", "venue_key"])

missing = (joined.isna().mean().mul(100).round(1)
                 .rename("% missing").to_frame()
                 .join(joined.dtypes.rename("dtype")))
display(missing.sort_values("% missing", ascending=False))

joined.to_csv(OUT_PATH, index=False)
print(f"Saved {OUT_PATH}: {len(joined):,} rows x {joined.shape[1]} columns")

,% missing,dtype
year_formed,86.7,float64
years_active,86.7,float64
artist_country,84.8,str
median_income,29.4,float64
state,26.1,str
venue_country,24.9,str
lon,24.7,float64
timezone,24.7,str
sg_upcoming_events,24.7,float64
is_us,24.7,object


Saved data/joined_table.csv: 29,613 rows x 38 columns


## 6. Quick look

In [6]:
joined.head()

,name,weekday,is_weekend,month,show_hour,days_to_show,listings,max_listings,n_priced_days,price_median,...,state,venue_country,is_us,lat,lon,timezone,venue_tier,median_max_listings,sg_upcoming_events,median_income
0,Mammoth,Monday,False,7,18,10,6,9,6,109.0,...,GA,US,True,32.0531,-81.1031,America/New_York,unknown,9.0,1.0,69575.0
1,Bumpin Uglies,Monday,False,7,18,10,1,1,6,41.0,...,NaN,NaN,NaN,NaN,NaN,NaN,unknown,1.0,NaN,NaN
2,Toast - The Best of Bread,Monday,False,7,18,10,60,60,6,80.0,...,MN,US,True,44.7744,-93.2792,America/Chicago,unknown,60.0,20.0,105212.0
3,Dawes Duo,Monday,False,7,18,10,2,5,6,478.0,...,NaN,NaN,NaN,NaN,NaN,NaN,unknown,5.0,NaN,NaN
4,Poppy,Monday,False,7,18,10,12,14,6,101.0,...,NaN,NaN,NaN,NaN,NaN,NaN,unknown,5.0,NaN,NaN
